In [ ]:
### Decoing 단계에서 1개의 입력을 처리하기 위해 로드해야 할 Matrix 메모리가 많음
    => memory-bound 인 decodeing(추론) 단계의 메모리 및 지연을 줄이는 목적 (모델 크기는 변화 없음)
    => KV Cache Pruning, Speculative Decoding

    * KV Cahce 메모리 계산
        KV Bytes = 2 x B x # of Layer x # of Head(kv) x dim of Head x Seq_length x dtype_bytes

### KV Cache Pruning
    - 긴 문맥에서 선형으로 커지는 KV Cache 를 중요한 토큰의 Key / Value 만 예산 안에 남기는 방법
    - StreamingLLM, H2O, SnapKV 등
    - 정확도 (passkey retrieval) vs 캐시 메모리 비교
    - attn_kv 의 shape
        - [ B, H, Q, K ]
            B : batch size
            H : head 개수
            Q : query token 개수
            K : KV cache 의 token 개수

    - scores 의 shape
        - [B, H, K]
            B : batch size          - b 번재 batch
            H : head 개수            - h 번째 head
            K : KV cache token 개수  - k 번째 KV token 의 중요도

    - Scoring 방식 (무엇이 중요한가)
        - StreamingLLM  : 위치만 사용
            - 맨앞 토큰 n 개 + 최근 윈도우 / 중간은 무시
        - H2O           : 누적 Attention Score
            - 누적 Attention Score 가 큰 top k 개 사용 + 최근 윈도우 (Attention 누적은 앞 토큰이 유리)
            - 누적 Attention Score 를 영향 받은 KV Cahce 토큰 개수로 normalize + 최근 윈도우로 개선
        - SnapKV        : 누적 Attention Score + observation window
            - 최근 윈도우(w)의 Query 가 KV token 에 준 Attention Score 만 합산
            - 각 토큰 주변 커널 범위에서 가장 큰 score 만 가져옴

### Speculative Decoding
    - 작은 draft 모델이 여러 토큰을 미리 제안
    - target 모델이 한 번의 forward 로 검증
    - chain -> tree -> parallel block 으로 drafting 방식을 확장
    - 출력은 AR decodeing 과 토큰 단위로 동일

In [ ]:
### StreamingLLM Scoring
    - 맨앞 토큰 + 최근 토큰

# =====================================================================
# 2-3. StreamingLLM Score
#   맨 앞 sink token + 최근 token을 유지합니다.
# =====================================================================

def streaming_score(attn_kv, layer, budget, n_sink=4):
    B, H, K, _ = layer.keys.shape

    # 최근 token일수록 높은 점수: 0, 1, 2, ..., K-1
    scores = torch.arange(K, device=DEVICE).float().repeat(B, H, 1)

    # 처음 n_sink개 token은 무조건 유지
    scores[:, :, :n_sink] = float("inf")

    return scores

In [ ]:
### H2O (Heavy-Hitter Oracle) Scoring
    - 누적 Attention Score + 최근 토큰
    - 누적 Attention Score / 영향 받은 kv 토큰 수 + 최근 토큰

# =====================================================================
# 2-5. H2O Score
#   전체 query의 attention을 누적하여 heavy-hitter token을 찾습니다.
# =====================================================================

def h2o_score(attn_kv, layer, budget):
    recent_size = budget//2

    # 전체 query의 attention 누적
    scores = attn_kv.sum(dim=-2)        # [B, H, Q, K] 중 Q 인 query 별로 attention 누적

    # 최근 recent_size개 token은 항상 보존
    scores[..., -recent_size:] = float("inf")

    return scores


# =====================================================================
# H2O의 구조적 편향: H2O vs H2O-Norm
# =====================================================================

def h2o_norm_score(attn_kv, layer, budget):
    """누적 attention을 실제로 attend된 query 개수로 나눈 변형."""
    B, H, Q, K = attn_kv.shape
    recent_size = budget // 2

    # 전체 query의 attention 누적
    scores = attn_kv.sum(dim=-2)

    # token j가 실제로 attention을 받을 수 있었던 query 수
    count = torch.arange(
        K, 0, -1,
        device=attn_kv.device,
        dtype=scores.dtype,
    )

    # 최근 token은 항상 보존
    scores[..., -recent_size:] = float("inf")

    return scores / count.view(1, 1, K)

In [ ]:
### SnapKV Scoring
    - 최근 윈도우(w)의 Query 가 KV token 에 준 Attention Score 누적 + 최근 토큰

# =====================================================================
# 2-7. SnapKV Score
#   마지막 observation window의 attention을 사용하고,
#   max pooling으로 인접 token의 중요도를 함께 반영합니다.
# =====================================================================

def snapkv_score(attn_kv, layer, budget, window=32, kernel=7):
    B, H, Q, K = attn_kv.shape
    w = min(window, Q)

    # 마지막 w개의 query가 각 KV token에 준 attention을 합산하세요.
    # Hint: attn_kv shape = [B, H, Q, K]
    scores = attn_kv[:, :, -w:, :].sum(dim=2)

    # 각 token 주변 kernel 범위에서 가장 큰 score를 가져옵니다.
    half = kernel // 2
    pooled_scores = scores.clone()

    for j in range(K):
        start = max(0, j - half)
        end   = min(K, j + half + 1)

        # Hint: start ~ end 위치 중 최대 score
        pooled_scores[:, :, j] = scores[:, :, start:end].max(dim=-1).values

    scores = pooled_scores

    # Observation window는 항상 유지
    scores[:, :, -w:] = float("inf")

    return scores